# Milestone 4 — Memory & RAG
### Stage 5 (part 2) of NOVA: Short-term Context, Long-term Memory, and semantic search

This notebook teaches the concepts behind `nova/memory/context_store.py`, `long_term_store.py`, `rag_store.py`, and `manager.py`. We'll cover:

1. Why NOVA uses three *different* memory stores instead of one
2. SQLite basics: Short-term Context and Long-term Memory
3. What an embedding actually is, and why vector similarity finds "meaning" not just matching words
4. ChromaDB and the RAG (Retrieval-Augmented Generation) pattern
5. How `MemoryManager` decides what to read/write and assembles context for the LLM

> This notebook runs fully in this sandbox for SQLite and the "toy embedding" sections. The real sentence-transformers + ChromaDB cells are marked "run this on your own machine" since downloading model weights needs internet this sandbox doesn't have.


In [1]:
# Install locally (not in this sandbox):
# pip install chromadb sentence-transformers

import sqlite3
import time
print("Ready.")


Ready.


## 1. Why three separate memory stores?

| Store | Scope | Lookup method | Example |
|---|---|---|---|
| **Short-term Context** | This conversation only | By session ID, most recent N turns | "what did I just ask you?" |
| **Long-term Memory** | Forever, across all sessions | Exact/keyword match | "what's my favorite color?" |
| **RAG Knowledge** | Forever, across all sessions | Semantic similarity | "what does this codebase's auth module do?" |

If you crammed all of this into one table, you'd hit real problems: clearing a conversation ("forget this chat") would risk deleting a stored preference too, and a keyword search over documents doesn't work well -- if you ask "how do I reset my password" but the document says "recovering account access," a keyword match misses it entirely. That last problem is exactly what embeddings solve.


## 2. Short-term Context & Long-term Memory (SQLite)

Both are plain SQL tables. The only meaningful design decision is **what the schema captures**:


In [2]:
# Short-term Context: every row is one conversation turn
conn = sqlite3.connect(":memory:")  # in-memory for this demo; the real one is a file on disk
conn.execute('''
    CREATE TABLE turns (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        session_id TEXT NOT NULL,
        role TEXT NOT NULL,
        content TEXT NOT NULL,
        created_at REAL NOT NULL
    )
''')

conn.execute("INSERT INTO turns (session_id, role, content, created_at) VALUES (?, ?, ?, ?)",
             ("session-1", "user", "hello", time.time()))
conn.execute("INSERT INTO turns (session_id, role, content, created_at) VALUES (?, ?, ?, ?)",
             ("session-1", "assistant", "hi there!", time.time()))
conn.commit()

rows = conn.execute("SELECT role, content FROM turns WHERE session_id = ? ORDER BY id", ("session-1",)).fetchall()
print(rows)


[('user', 'hello'), ('assistant', 'hi there!')]


In [3]:
# Long-term Memory: every row is one durable fact, deduplicated via UNIQUE
ltm_conn = sqlite3.connect(":memory:")
ltm_conn.execute('''
    CREATE TABLE facts (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        fact TEXT NOT NULL UNIQUE,
        created_at REAL NOT NULL
    )
''')

def remember(fact):
    ltm_conn.execute("INSERT OR IGNORE INTO facts (fact, created_at) VALUES (?, ?)", (fact, time.time()))
    ltm_conn.commit()

remember("my favorite color is blue")
remember("my favorite color is blue")  # duplicate -- silently ignored thanks to UNIQUE + INSERT OR IGNORE
remember("my dog's name is Rex")

print(ltm_conn.execute("SELECT fact FROM facts").fetchall())


[("my dog's name is Rex",), ('my favorite color is blue',)]


Notice `INSERT OR IGNORE` combined with the `UNIQUE` constraint on `fact` -- this is the entire deduplication mechanism. Say "remember that my favorite color is blue" to NOVA five times and it's still stored exactly once.


## 3. What is an embedding, actually?

An embedding model turns text into a list of numbers (a vector) such that texts with **similar meaning** end up as **nearby vectors** -- even if they don't share any of the same words. This is the whole trick behind semantic search.

We can't run the real `sentence-transformers` model in this sandbox (no internet to download weights), but let's build a *toy* embedding to build intuition: represent each word as present (1) or absent (0) across a small fixed vocabulary, then measure similarity by how many words overlap.


In [4]:
import math

vocabulary = ["password", "reset", "account", "access", "recover", "login", "email", "weather", "today"]

def toy_embed(text):
    words = set(text.lower().split())
    return [1.0 if v in words else 0.0 for v in vocabulary]

def cosine_similarity(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x * x for x in a))
    norm_b = math.sqrt(sum(x * x for x in b))
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot / (norm_a * norm_b)

query = "how do I reset my password"
doc_a = "recovering account access after losing your login"
doc_b = "what is the weather today"

q_vec = toy_embed(query)
a_vec = toy_embed(doc_a)
b_vec = toy_embed(doc_b)

print(f"query vs doc_a (about account recovery): {cosine_similarity(q_vec, a_vec):.2f}")
print(f"query vs doc_b (about weather):           {cosine_similarity(q_vec, b_vec):.2f}")


query vs doc_a (about account recovery): 0.00
query vs doc_b (about weather):           0.00


With this toy embedding (literal word overlap), `doc_a` and the query share zero exact words ("reset password" vs "recovering account access") so similarity comes out at 0 -- this toy version fails exactly the case real embeddings are good at.

A real sentence-transformers model doesn't look at literal words -- it was trained on millions of sentence pairs to place *semantically similar* sentences near each other in vector space, regardless of exact wording. That's why "how do I reset my password" and "recovering account access" end up close together with a real model, but not with our toy word-overlap version.


In [5]:
# --- RUN THIS ON YOUR OWN MACHINE ---
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")  # matches config.yaml's embedding_model

query_vec = model.encode("how do I reset my password")
doc_a_vec = model.encode("recovering account access after losing your login")
doc_b_vec = model.encode("what is the weather today")

def cosine_sim(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

print("query vs doc_a:", cosine_sim(query_vec, doc_a_vec))  # expect notably higher
print("query vs doc_b:", cosine_sim(query_vec, doc_b_vec))  # expect notably lower


c:\Users\ashok\OneDrive\Documents\VSCODE\NOVA-Assistant\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 42663.77it/s]


query vs doc_a: 0.5187502503395081
query vs doc_b: -0.00032315216958522797


## 4. ChromaDB and the RAG pattern

RAG = **Retrieval-Augmented Generation**: instead of the LLM answering purely from what it learned during training, you *retrieve* relevant text first, then hand that text to the LLM as extra context. This is exactly why the diagram shows RAG Knowledge feeding into the Brain before the Final Answer stage.

ChromaDB stores `(id, embedding, document, metadata)` tuples and, given a query embedding, returns the nearest stored documents by vector distance.


In [6]:
# --- RUN THIS ON YOUR OWN MACHINE ---
import chromadb

client = chromadb.PersistentClient(path="./data/chroma")  # matches config.yaml's persist_directory
collection = client.get_or_create_collection("nova_knowledge")

# Indexing a document (this is what RagKnowledgeStore.add_document does)
doc_text = "NOVA is a modular, open-source, AI-powered voice assistant for Windows."
embedding = model.encode(doc_text).tolist()
collection.upsert(ids=["doc1"], embeddings=[embedding], documents=[doc_text], metadatas=[{}])

# Querying (this is what RagKnowledgeStore.query does)
query_embedding = model.encode("what is NOVA?").tolist()
results = collection.query(query_embeddings=[query_embedding], n_results=1)
print(results["documents"][0])


ValueError: Expected metadata to be a non-empty dict, got 0 metadata attributes in upsert.

### Why this is separate from Long-term Memory

Long-term Memory answers "what did the user tell me directly" (facts, preferences) via exact lookup. RAG Knowledge answers "what's in this document/codebase/notes I fed you" via semantic search over much larger, unstructured text. Mixing them would mean either running expensive vector search for a simple key-value lookup, or losing semantic search over documents by forcing exact-match lookups on them.


## 5. MemoryManager: deciding what to read/write

`MemoryManager` is the piece that decides, on every turn: *should this become a long-term fact?* and *what should I retrieve before answering?* It's deliberately simple right now -- a keyword trigger for writes, and "always fetch facts + top RAG hits + recent turns" for reads.


In [ ]:
REMEMBER_TRIGGERS = ("remember that", "remember this", "don't forget", "note that")

def maybe_extract_fact(content):
    lowered = content.lower()
    for trigger in REMEMBER_TRIGGERS:
        idx = lowered.find(trigger)
        if idx != -1:
            return content[idx + len(trigger):].strip(" .")
    return None

print(maybe_extract_fact("remember that my favorite color is blue"))
print(maybe_extract_fact("what's the weather like today?"))  # None -- no trigger phrase


### Exercise: a smarter trigger

The keyword approach above misses "my favorite color is blue, please remember" (trigger phrase at the *end*) and "I like blue" (no trigger phrase at all, but arguably worth remembering). Try writing a version that uses the LLM itself to classify "is this worth remembering as a long-term fact? If so, what's the fact?" -- and think about the tradeoff: an LLM call on every single turn costs more time/compute than a keyword check, for a benefit that only matters some of the time.

### Exercises
1. Build a tiny end-to-end demo: store 3 "documents" using the toy embedding from Section 3, then query with a phrase that shares no exact words with your target document -- confirm the toy embedding fails to find it (this is expected -- it's why we don't use toy embeddings in production).
2. Once you have `sentence-transformers` installed locally, repeat exercise 1 with the real model and confirm it *does* find the right document.
3. Add a `session_id` isolation test: store facts and short-term turns for two different sessions, and confirm `get_recent()` never leaks session A's turns into session B's context.

**You now have the full picture of Stage 5.** Next up: Stage 6 — Tool Modules (Windows automation with pyautogui/pywinauto, browser automation with Playwright).
